# Domain-Specific Tokenization and Chunk-Aware Preprocessing

This notebook starts implementation for improving information extraction using domain-aware text preprocessing before chunking.

Focus areas:
- domain-specific tokenization for finance and email text
- chunk-aware preprocessing and BIO consistency checks
- measurable diagnostics for alignment and output health

## 1. Set Up Notebook Environment

Import dependencies, set display options, and define reusable helpers for path and reproducibility.

In [13]:
from __future__ import annotations

import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)


def find_project_root(start: Path) -> Path:
    """Find the project root by locating expected top-level folders."""
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / "dataset").exists() and (cur / "ipynb").exists():
            return cur
        cur = cur.parent
    return start.resolve()


PROJECT_ROOT = find_project_root(Path.cwd())
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Seed:", SEED)

Project root: /home/sg/dev/nlp
Seed: 42


## 2. Define Project Configuration

Keep paths and runtime options in one place so experiments are easy to tune and reproduce.

In [14]:
CHUNK_LABELS = [
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP", "B-INTJ", "I-INTJ", "B-LST",
    "I-LST", "B-NP", "I-NP", "B-PP", "I-PP", "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP",
    "B-VP", "I-VP",
]


@dataclass(frozen=True)
class Config:
    data_dir: Path
    train_path: Path
    test_path: Path
    sample_size: int = 250
    model_checkpoint: str = "distilbert/distilbert-base-uncased"
    use_model_tokenizer_diagnostics: bool = True


CFG = Config(
    data_dir=PROJECT_ROOT / "dataset",
    train_path=PROJECT_ROOT / "dataset" / "train.parquet",
    test_path=PROJECT_ROOT / "dataset" / "test.parquet",
)

print(CFG)

Config(data_dir=PosixPath('/home/sg/dev/nlp/dataset'), train_path=PosixPath('/home/sg/dev/nlp/dataset/train.parquet'), test_path=PosixPath('/home/sg/dev/nlp/dataset/test.parquet'), sample_size=250, model_checkpoint='distilbert/distilbert-base-uncased', use_model_tokenizer_diagnostics=True)


## 3. Implement Core Functions

Implement reusable functions for data loading, domain-specific tokenization, chunk-aware preprocessing, and diagnostics.

In [15]:
def read_parquet_safe(path: Path) -> pd.DataFrame:
    """Read parquet with engine fallback for environment compatibility."""
    last_error: Optional[Exception] = None
    for engine in ("fastparquet", "pyarrow"):
        try:
            return pd.read_parquet(path, engine=engine)
        except Exception as exc:  # pragma: no cover - notebook runtime guard
            last_error = exc
    raise RuntimeError(f"Could not read parquet file: {path}") from last_error


def decode_chunk_ids(chunk_ids: Sequence[int]) -> List[str]:
    """Convert integer chunk IDs to label strings."""
    return [CHUNK_LABELS[int(i)] for i in chunk_ids]


def load_conll_frames(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load train/test CoNLL-style frames and attach decoded chunk labels."""
    train_df = read_parquet_safe(cfg.train_path).copy()
    test_df = read_parquet_safe(cfg.test_path).copy()

    train_df["chunk_labels"] = train_df["chunk_tags"].apply(decode_chunk_ids)
    test_df["chunk_labels"] = test_df["chunk_tags"].apply(decode_chunk_ids)
    return train_df, test_df

In [16]:
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+\b", re.IGNORECASE)
CURRENCY_AMOUNT_RE = re.compile(r"(?<!\w)([$€£])\s?(\d+(?:[.,]\d+)?)(\s?(?:k|m|b|million|billion))?", re.IGNORECASE)
TIME_RE = re.compile(r"\b(?:[01]?\d|2[0-3]):[0-5]\d(?:\s?(?:am|pm))?\b", re.IGNORECASE)
DATE_RE = re.compile(r"\b\d{1,2}[-/]\d{1,2}(?:[-/]\d{2,4})?\b")
ID_RE = re.compile(r"\b(?:ID|SKU|REF|TKT)[-:#][A-Z0-9]{3,}\b", re.IGNORECASE)


def normalize_domain_text(text: str) -> str:
    """Normalize high-value domain tokens into canonical single-token forms."""
    out = text.strip()
    out = EMAIL_RE.sub(lambda m: f"EMAIL<{m.group(0).lower()}>", out)
    out = URL_RE.sub(lambda m: f"URL<{m.group(0).lower()}>", out)
    out = CURRENCY_AMOUNT_RE.sub(
        lambda m: f"{m.group(1)}{m.group(2)}{(m.group(3) or '').replace(' ', '')}",
        out,
    )
    out = TIME_RE.sub(lambda m: f"TIME<{m.group(0).lower().replace(' ', '')}>", out)
    out = DATE_RE.sub(lambda m: f"DATE<{m.group(0)}>", out)
    out = ID_RE.sub(lambda m: f"ID<{m.group(0).upper()}>", out)
    return re.sub(r"\s+", " ", out).strip()


def custom_word_tokenize(text: str) -> List[str]:
    """Whitespace-first tokenization preserving protected tags like EMAIL<...>."""
    if not text:
        return []
    return text.split(" ")


def sentence_from_tokens(tokens: Sequence[str]) -> str:
    """Create a sentence string from token list for processing demos."""
    return " ".join(tokens).strip()

In [17]:
def validate_bio_sequence(labels: Sequence[str]) -> bool:
    """Return True when BIO transitions are valid for chunk labels."""
    prev_type = None
    prev_prefix = "O"
    for label in labels:
        if label == "O":
            prev_type, prev_prefix = None, "O"
            continue
        if "-" not in label:
            return False
        prefix, chunk_type = label.split("-", 1)
        if prefix not in {"B", "I"}:
            return False
        if prefix == "I" and not (
            prev_prefix in {"B", "I"} and prev_type == chunk_type
        ):
            return False
        prev_type, prev_prefix = chunk_type, prefix
    return True


def fix_bio_sequence(labels: Sequence[str]) -> List[str]:
    """Repair invalid I-* transitions by converting them to B-*."""
    fixed: List[str] = []
    prev_type = None
    prev_prefix = "O"
    for label in labels:
        if label == "O" or "-" not in label:
            fixed.append("O" if label == "O" else label)
            prev_type, prev_prefix = (None, "O") if label == "O" else (prev_type, prev_prefix)
            continue
        prefix, chunk_type = label.split("-", 1)
        if prefix == "I" and not (prev_prefix in {"B", "I"} and prev_type == chunk_type):
            fixed_label = f"B-{chunk_type}"
        else:
            fixed_label = label
        fixed.append(fixed_label)
        prev_prefix, prev_type = fixed_label.split("-", 1)
    return fixed


def tokenizer_alignment_diagnostics(
    token_sequences: Sequence[Sequence[str]],
    model_checkpoint: str,
) -> Dict[str, float]:
    """Measure how often model tokenization can align with custom token sequences."""
    try:
        from transformers import AutoTokenizer
    except Exception:
        return {
            "examples": float(len(token_sequences)),
            "alignment_rate": np.nan,
            "valid_word_id_rate": np.nan,
            "tokenizer_error": "transformers_not_installed",
        }

    tokenizer_kwargs = {}
    if "roberta" in model_checkpoint.lower():
        tokenizer_kwargs["add_prefix_space"] = True

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, **tokenizer_kwargs)
    except Exception as exc:
        return {
            "examples": float(len(token_sequences)),
            "alignment_rate": np.nan,
            "valid_word_id_rate": np.nan,
            "tokenizer_error": str(exc)[:200],
        }

    aligned_examples = 0
    total_word_ids = 0
    valid_word_ids = 0

    for words in token_sequences:
        encoded = tokenizer(list(words), truncation=True, is_split_into_words=True)
        word_ids = encoded.word_ids()
        if word_ids is None:
            continue
        aligned_examples += 1
        total_word_ids += len(word_ids)
        valid_word_ids += sum(1 for x in word_ids if x is not None)

    return {
        "examples": float(len(token_sequences)),
        "alignment_rate": aligned_examples / max(len(token_sequences), 1),
        "valid_word_id_rate": valid_word_ids / max(total_word_ids, 1),
        "tokenizer_error": "",
    }

## 4. Create a Minimal Execution Pipeline

Wire the core functions end-to-end on sample data and store intermediate outputs for inspection.

In [18]:
train_df, test_df = load_conll_frames(CFG)

sample_df = test_df.head(CFG.sample_size).copy()
sample_df["raw_sentence"] = sample_df["tokens"].apply(sentence_from_tokens)
sample_df["normalized_sentence"] = sample_df["raw_sentence"].apply(normalize_domain_text)
sample_df["raw_token_count"] = sample_df["tokens"].apply(len)
sample_df["normalized_tokens"] = sample_df["normalized_sentence"].apply(custom_word_tokenize)
sample_df["normalized_token_count"] = sample_df["normalized_tokens"].apply(len)
sample_df["token_delta"] = sample_df["normalized_token_count"] - sample_df["raw_token_count"]

alignment_report = tokenizer_alignment_diagnostics(
    sample_df["normalized_tokens"].tolist(),
    model_checkpoint=CFG.model_checkpoint,
)

display(
    sample_df[
        [
            "raw_sentence",
            "normalized_sentence",
            "raw_token_count",
            "normalized_token_count",
            "token_delta",
        ]
    ].head(12)
)
print("Alignment report:", alignment_report)

,raw_sentence,normalized_sentence,raw_token_count,normalized_token_count,token_delta
0,Rockwell International Corp. 's Tulsa unit said it signed a tentative agreement extending its contract with Boeing Co. to provide structural parts for Boeing 's 747 jetliners .,Rockwell International Corp. 's Tulsa unit said it signed a tentative agreement extending its contract with Boeing Co. to provide structural parts for Boeing 's 747 jetliners .,28,28,0
1,Rockwell said the agreement calls for it to supply 200 additional so-called shipsets for the planes .,Rockwell said the agreement calls for it to supply 200 additional so-called shipsets for the planes .,17,17,0
2,"These include , among other parts , each jetliner 's two major bulkheads , a pressure floor , torque box , fixed leading edges for the wings and an aft keel beam .","These include , among other parts , each jetliner 's two major bulkheads , a pressure floor , torque box , fixed leading edges for the wings and an aft keel beam .",33,33,0
3,"Under the existing contract , Rockwell said , it has already delivered 793 of the shipsets to Boeing .","Under the existing contract , Rockwell said , it has already delivered 793 of the shipsets to Boeing .",19,19,0
4,"Rockwell , based in El Segundo , Calif. , is an aerospace , electronics , automotive and graphics concern .","Rockwell , based in El Segundo , Calif. , is an aerospace , electronics , automotive and graphics concern .",20,20,0
5,"Frank Carlucci III was named to this telecommunications company 's board , filling the vacancy created by the death of William Sobey last May .","Frank Carlucci III was named to this telecommunications company 's board , filling the vacancy created by the death of William Sobey last May .",25,25,0
6,"Mr. Carlucci , 59 years old , served as defense secretary in the Reagan administration .","Mr. Carlucci , 59 years old , served as defense secretary in the Reagan administration .",16,16,0
7,"In January , he accepted the position of vice chairman of Carlyle Group , a merchant banking concern .","In January , he accepted the position of vice chairman of Carlyle Group , a merchant banking concern .",19,19,0
8,SHEARSON LEHMAN HUTTON Inc .,SHEARSON LEHMAN HUTTON Inc .,5,5,0
9,"Thomas E. Meador , 42 years old , was named president and chief operating officer of Balcor Co. , a Skokie , Ill. , subsidiary of this New York investment banking firm .","Thomas E. Meador , 42 years old , was named president and chief operating officer of Balcor Co. , a Skokie , Ill. , subsidiary of this New York investment banking firm .",33,33,0


Alignment report: {'examples': 250.0, 'alignment_rate': 1.0, 'valid_word_id_rate': 0.9362570117287098, 'tokenizer_error': ''}


## 5. Run Validation Checks

Run sanity checks on dataset integrity and transformation outputs.

In [19]:
validation_summary = {
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "sample_rows": int(len(sample_df)),
    "null_raw_sentence": int(sample_df["raw_sentence"].isna().sum()),
    "null_normalized_sentence": int(sample_df["normalized_sentence"].isna().sum()),
    "mean_raw_tokens": float(sample_df["raw_token_count"].mean()),
    "mean_normalized_tokens": float(sample_df["normalized_token_count"].mean()),
    "mean_token_delta": float(sample_df["token_delta"].mean()),
    "bio_valid_rate": float(
        sample_df["chunk_labels"].apply(validate_bio_sequence).mean()
    ),
}

validation_df = pd.DataFrame([validation_summary])
display(validation_df)

print("Validation checks complete.")

,train_rows,test_rows,sample_rows,null_raw_sentence,null_normalized_sentence,mean_raw_tokens,mean_normalized_tokens,mean_token_delta,bio_valid_rate
0,8937,2013,250,0,0,24.008,23.56,-0.448,1.0


Validation checks complete.


## 6. Add Unit-Test Style Assertions

Use lightweight assertions to lock in deterministic behavior of normalization and BIO preprocessing logic.

In [20]:
sample_text = "Email ops@acme.com before 04/21/2026 at 3:30 PM and include SKU-AB12X9 in report."
norm_1 = normalize_domain_text(sample_text)
norm_2 = normalize_domain_text(sample_text)

assert norm_1 == norm_2, "Normalization should be deterministic"
assert "EMAIL<ops@acme.com>" in norm_1, "Email should be canonicalized"
assert "DATE<04/21/2026>" in norm_1, "Date should be canonicalized"
assert "TIME<3:30pm>" in norm_1, "Time should be canonicalized"
assert "ID<SKU-AB12X9>" in norm_1, "ID token should be canonicalized"

false_id_text = "These include details for review."
assert "ID<INCLUDE>" not in normalize_domain_text(false_id_text), "Common words must not be ID-tagged"

false_time_text = "The value is $ 2.48 a share."
assert "TIME<2.48>" not in normalize_domain_text(false_time_text), "Currency decimals must not be treated as time"

invalid_bio = ["I-NP", "I-NP", "O", "I-VP", "B-NP"]
fixed_bio = fix_bio_sequence(invalid_bio)
assert validate_bio_sequence(fixed_bio), "Fixed BIO sequence should be valid"
assert fixed_bio[0] == "B-NP", "First invalid I-* should convert to B-*"
assert fixed_bio[3] == "B-VP", "Invalid I-VP after O should convert to B-VP"

print("All assertion checks passed.")

All assertion checks passed.


## 7. Run Ablation Evaluation

Execute four planned variants on a fixed subset using a local token-classification checkpoint:

- baseline
- tokenization_only
- tokenization_plus_bio
- tokenization_bio_normalization

For each variant, we track:

- alignment-aware evaluable coverage
- chunk metrics on aligned token spans
- latency overhead

In [21]:
import importlib
import sys
import time
from difflib import SequenceMatcher
from typing import Any

DASHBOARD_SRC = PROJECT_ROOT / "libs" / "chunk_event_dashboard" / "src"
if DASHBOARD_SRC.exists() and str(DASHBOARD_SRC) not in sys.path:
    sys.path.insert(0, str(DASHBOARD_SRC))

DASHBOARD_IMPORT_ERROR = ""
DASHBOARD_READY = False

try:
    _inference_module = importlib.import_module("chunk_event_dashboard.inference")
    load_token_classifier = getattr(_inference_module, "load_token_classifier")
    merge_subword_predictions = getattr(_inference_module, "merge_subword_predictions")
    DASHBOARD_READY = True
except Exception as exc:
    DASHBOARD_IMPORT_ERROR = str(exc)

try:
    import evaluate

    _SEQEVAL = evaluate.load("seqeval")
except Exception:
    _SEQEVAL = None

VARIANTS = [
    "baseline",
    "tokenization_only",
    "tokenization_plus_bio",
    "tokenization_bio_normalization",
]

ALIGNMENT_MIN_COVERAGE = 0.60


def preprocess_variant(tokens: Sequence[str], labels: Sequence[str], variant: str) -> Dict[str, Any]:
    raw_sentence = sentence_from_tokens(tokens)

    if variant == "baseline":
        processed_sentence = raw_sentence
        processed_tokens = list(tokens)
        processed_labels = list(labels)
    elif variant == "tokenization_only":
        processed_sentence = normalize_domain_text(raw_sentence)
        processed_tokens = custom_word_tokenize(processed_sentence)
        processed_labels = list(labels)
    elif variant == "tokenization_plus_bio":
        processed_sentence = raw_sentence
        processed_tokens = custom_word_tokenize(processed_sentence)
        processed_labels = fix_bio_sequence(labels)
    elif variant == "tokenization_bio_normalization":
        processed_sentence = normalize_domain_text(raw_sentence)
        processed_tokens = custom_word_tokenize(processed_sentence)
        processed_labels = fix_bio_sequence(labels)
    else:
        raise ValueError(f"Unknown variant: {variant}")

    return {
        "processed_sentence": processed_sentence,
        "processed_tokens": processed_tokens,
        "processed_labels": processed_labels,
    }


def _safe_chunk_label(label: str) -> str:
    return label if label in CHUNK_LABELS else "O"


def _normalize_alignment_token(token: str) -> str:
    raw = str(token).strip().lower()
    raw = raw.replace("’", "'").replace("`", "'")
    stripped = re.sub(r"^[\W_]+|[\W_]+$", "", raw)
    return stripped if stripped else raw


def align_labels_by_tokens(
    ref_tokens: Sequence[str],
    ref_labels: Sequence[str],
    pred_tokens: Sequence[str],
    pred_labels: Sequence[str],
) -> Tuple[List[str], List[str], float]:
    """Align reference and predicted labels by token text similarity, not strict length."""
    ref_norm = [_normalize_alignment_token(tok) for tok in ref_tokens]
    pred_norm = [_normalize_alignment_token(tok) for tok in pred_tokens]

    matcher = SequenceMatcher(a=ref_norm, b=pred_norm, autojunk=False)

    aligned_true: List[str] = []
    aligned_pred: List[str] = []
    covered_ref_indices = set()

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            continue
        block_len = min(i2 - i1, j2 - j1)
        for off in range(block_len):
            ref_idx = i1 + off
            pred_idx = j1 + off
            if ref_idx >= len(ref_labels) or pred_idx >= len(pred_labels):
                continue
            aligned_true.append(ref_labels[ref_idx])
            aligned_pred.append(_safe_chunk_label(pred_labels[pred_idx]))
            covered_ref_indices.add(ref_idx)

    coverage = len(covered_ref_indices) / max(len(ref_labels), 1)
    return aligned_true, aligned_pred, coverage


def predict_chunk_labels(pipe, sentence: str) -> Tuple[List[str], List[str], List[float]]:
    pred_items = pipe(sentence)
    pred_tokens, pred_labels, pred_scores = merge_subword_predictions(pred_items)
    pred_labels = [_safe_chunk_label(tag) for tag in pred_labels]
    return pred_tokens, pred_labels, pred_scores


def compute_chunk_metrics(true_sequences: List[List[str]], pred_sequences: List[List[str]]) -> Dict[str, float]:
    if len(true_sequences) == 0:
        return {
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "accuracy": np.nan,
        }

    if _SEQEVAL is not None:
        out = _SEQEVAL.compute(predictions=pred_sequences, references=true_sequences)
        return {
            "precision": float(out.get("overall_precision", np.nan)),
            "recall": float(out.get("overall_recall", np.nan)),
            "f1": float(out.get("overall_f1", np.nan)),
            "accuracy": float(out.get("overall_accuracy", np.nan)),
        }

    total = 0
    correct = 0
    for t_seq, p_seq in zip(true_sequences, pred_sequences):
        n = min(len(t_seq), len(p_seq))
        total += n
        correct += sum(1 for i in range(n) if t_seq[i] == p_seq[i])

    return {
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "accuracy": correct / max(total, 1),
    }


ablation_detail_df = pd.DataFrame()
ablation_metrics_df = pd.DataFrame()
model_pipeline = None
checkpoint_path = None

if not DASHBOARD_READY:
    print("Dashboard modules unavailable. Skipping model-backed ablation run.")
    print("Import error:", DASHBOARD_IMPORT_ERROR)
else:
    model_pipeline, checkpoint_path = load_token_classifier("distilbert", PROJECT_ROOT)
    print("Loaded checkpoint:", checkpoint_path)

    ablation_eval_df = test_df.head(min(max(CFG.sample_size, 120), len(test_df))).copy()

    detail_rows = []
    metric_rows = []

    for variant in VARIANTS:
        y_true = []
        y_pred = []
        variant_start = time.perf_counter()

        for _, row in ablation_eval_df.iterrows():
            pre = preprocess_variant(row["tokens"], row["chunk_labels"], variant)
            pred_tokens, pred_labels, pred_scores = predict_chunk_labels(model_pipeline, pre["processed_sentence"])

            aligned_true, aligned_pred, alignment_coverage = align_labels_by_tokens(
                pre["processed_tokens"],
                pre["processed_labels"],
                pred_tokens,
                pred_labels,
            )

            length_match = len(pred_labels) == len(pre["processed_labels"])
            evaluable_row = alignment_coverage >= ALIGNMENT_MIN_COVERAGE and len(aligned_true) > 0

            if evaluable_row:
                y_true.append(aligned_true)
                y_pred.append(aligned_pred)

            detail_rows.append(
                {
                    "variant": variant,
                    "raw_sentence": sentence_from_tokens(row["tokens"]),
                    "processed_sentence": pre["processed_sentence"],
                    "raw_token_count": int(len(row["tokens"])),
                    "processed_token_count": int(len(pre["processed_tokens"])),
                    "pred_token_count": int(len(pred_tokens)),
                    "length_match": bool(length_match),
                    "alignment_coverage": float(alignment_coverage),
                    "evaluable_row": bool(evaluable_row),
                    "mean_pred_score": float(np.mean(pred_scores)) if pred_scores else np.nan,
                    "bio_valid_before": bool(validate_bio_sequence(row["chunk_labels"])),
                    "bio_valid_after": bool(validate_bio_sequence(pre["processed_labels"])),
                }
            )

        elapsed = time.perf_counter() - variant_start
        seq_metrics = compute_chunk_metrics(y_true, y_pred)

        metric_rows.append(
            {
                "variant": variant,
                "rows_total": int(len(ablation_eval_df)),
                "rows_evaluable": int(len(y_true)),
                "evaluable_rate": float(len(y_true) / max(len(ablation_eval_df), 1)),
                "mean_latency_ms": float(1000.0 * elapsed / max(len(ablation_eval_df), 1)),
                **seq_metrics,
            }
        )

    ablation_detail_df = pd.DataFrame(detail_rows)

    length_match_rate = (
        ablation_detail_df.groupby("variant")["length_match"].mean().rename("length_match_rate").reset_index()
    )
    alignment_coverage_mean = (
        ablation_detail_df.groupby("variant")["alignment_coverage"]
        .mean()
        .rename("mean_alignment_coverage")
        .reset_index()
    )
    alignment_evaluable_rate = (
        ablation_detail_df.groupby("variant")["evaluable_row"]
        .mean()
        .rename("alignment_evaluable_rate")
        .reset_index()
    )

    ablation_metrics_df = pd.DataFrame(metric_rows)
    ablation_metrics_df = ablation_metrics_df.merge(length_match_rate, on="variant", how="left")
    ablation_metrics_df = ablation_metrics_df.merge(alignment_coverage_mean, on="variant", how="left")
    ablation_metrics_df = ablation_metrics_df.merge(alignment_evaluable_rate, on="variant", how="left")
    ablation_metrics_df = ablation_metrics_df.sort_values("f1", ascending=False, na_position="last")

    display(ablation_metrics_df.round(4))
    print("Rows with low alignment coverage (sample):")
    display(ablation_detail_df[ablation_detail_df["alignment_coverage"] < ALIGNMENT_MIN_COVERAGE].head(10))

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 6793.58it/s]


Loaded checkpoint: /home/sg/dev/nlp/outputs/distilbert-conll2000/checkpoint-1677


/home/sg/dev/nlp/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


,variant,rows_total,rows_evaluable,evaluable_rate,mean_latency_ms,precision,recall,f1,accuracy,length_match_rate,mean_alignment_coverage,alignment_evaluable_rate
0,baseline,250,246,0.984,5.8967,0.9518,0.9605,0.9562,0.9709,0.116,0.8218,0.984
2,tokenization_plus_bio,250,246,0.984,6.5883,0.9518,0.9605,0.9562,0.9709,0.116,0.8218,0.984
1,tokenization_only,250,237,0.948,6.5385,0.8975,0.9152,0.9063,0.9227,0.116,0.8050,0.948
3,tokenization_bio_normalization,250,237,0.948,6.0072,0.8975,0.9152,0.9063,0.9227,0.116,0.8050,0.948


Rows with low alignment coverage (sample):


,variant,raw_sentence,processed_sentence,raw_token_count,processed_token_count,pred_token_count,length_match,alignment_coverage,evaluable_row,mean_pred_score,bio_valid_before,bio_valid_after
66,baseline,Warner-Lambert Co .,Warner-Lambert Co .,3,3,4,False,0.333333,False,0.992010,True,True
81,baseline,"Sales of Prozac , an anti-depressant , led drug-sales increases .","Sales of Prozac , an anti-depressant , led drug-sales increases .",11,11,12,False,0.545455,False,0.994388,True,True
97,baseline,"`` But it 's not mediocre , it 's a real problem . ''","`` But it 's not mediocre , it 's a real problem . ''",14,14,10,False,0.571429,False,0.943976,True,True
122,baseline,Clearly not .,Clearly not .,3,3,1,False,0.333333,False,0.955811,True,True
265,tokenization_only,"Great American Bank , citing depressed Arizona real estate prices , posted a third-quarter loss of $ 59.4 million , or $ 2.48 a share .","Great American Bank , citing depressed Arizona real estate prices , posted a third-quarter loss of $59.4million , or $2.48 a share .",26,23,26,True,0.576923,False,0.992548,True,True
267,tokenization_only,"For the nine months , it had a loss of $ 58.3 million , or $ 2.44 a share , after earnings of $ 29.5 million , or $ 1.20 a share , in the 1988 period .","For the nine months , it had a loss of $58.3million , or $2.44 a share , after earnings of $29.5million , or $1.20 a share , in the 1988 period .",38,32,36,False,0.526316,False,0.992097,True,True
279,tokenization_only,"In the $ 300-a-share buyout , that totaled about $ 76.7 million .","In the $300-a-share buyout , that totaled about $76.7million .",13,10,16,False,0.538462,False,0.987265,True,True
283,tokenization_only,"His 1988 salary was $ 575,000 , with a $ 575,000 bonus .","His 1988 salary was $575,000 , with a $575,000bonus .",13,10,14,False,0.538462,False,0.966607,True,True
287,tokenization_only,"That came to a combined $ 37.7 million under the $ 300-a-share buy-out , but just $ 21.3 million at yesterday 's close .","That came to a combined $37.7million under the $300-a-share buy-out , but just $21.3million at yesterday 's close .",24,19,29,False,0.500000,False,0.962826,True,True
311,tokenization_only,Sales for the quarter rose to $ 1.63 billion from $ 1.47 billion .,Sales for the quarter rose to $1.63billion from $1.47billion .,14,10,15,False,0.571429,False,0.992461,True,True


## 8. Measure Downstream IE Impact

Use the same model pipeline to estimate how each preprocessing variant affects event extraction behavior:

- accepted event rate
- confidence profile
- role completeness proxy
- estimated cost per accepted event

In [22]:
import importlib


def preprocess_free_text_variant(text: str, variant: str) -> str:
    if variant in {"tokenization_only", "tokenization_bio_normalization"}:
        return normalize_domain_text(text)
    return text


ie_detail_df = pd.DataFrame()
ie_summary_df = pd.DataFrame()

if model_pipeline is None or not DASHBOARD_READY:
    print("Skipping IE impact run because model pipeline is unavailable.")
else:
    try:
        _extraction_module = importlib.import_module("chunk_event_dashboard.extraction")
        extract_event_record = getattr(_extraction_module, "extract_event_record")
        estimate_event_cost = getattr(_extraction_module, "estimate_event_cost")
    except Exception as exc:
        print("Skipping IE impact run because extraction module is unavailable.")
        print("Import error:", str(exc))
    else:
        extra_domain_sentences = [
            "Please email finance@acme.com before 04/30/2026 5:00 PM with invoice ID-9XZ12.",
            "Meeting moved to 10:30 am at Building-7 room 204.",
            "The outage incident started at 03/12/2026 21:40 and resolved by 23:10.",
            "Submit budget update of $12.5k by tomorrow 9:00 AM.",
        ]

        base_sentences = sample_df["raw_sentence"].head(30).tolist()
        ie_input_sentences = base_sentences + extra_domain_sentences

        ie_rows = []
        for variant in VARIANTS:
            for sentence in ie_input_sentences:
                processed_sentence = preprocess_free_text_variant(sentence, variant)
                rec = extract_event_record(processed_sentence, model_pipeline, confidence_threshold=0.55)

                completeness_proxy = np.mean(
                    [
                        bool(rec.get("subject")),
                        bool(rec.get("trigger")),
                        bool(rec.get("object")),
                        bool(rec.get("location")),
                        bool(rec.get("time")),
                    ]
                )

                accepted = not bool(rec.get("abstained", True))
                est_cost = estimate_event_cost(int(rec.get("tokens_processed", 0)), "distilbert")

                ie_rows.append(
                    {
                        "variant": variant,
                        "original_sentence": sentence,
                        "processed_sentence": processed_sentence,
                        "event_type": rec.get("event_type"),
                        "accepted": accepted,
                        "abstain_reason": rec.get("abstain_reason"),
                        "confidence": float(rec.get("confidence", np.nan)),
                        "tokens_processed": int(rec.get("tokens_processed", 0)),
                        "cost_estimate": float(est_cost) if est_cost == est_cost else np.nan,
                        "role_completeness_proxy": float(completeness_proxy),
                    }
                )

        ie_detail_df = pd.DataFrame(ie_rows)

        ie_summary_df = (
            ie_detail_df.groupby("variant", as_index=False)
            .agg(
                total_records=("accepted", "size"),
                accepted_rate=("accepted", "mean"),
                mean_confidence=("confidence", "mean"),
                mean_tokens=("tokens_processed", "mean"),
                mean_cost_estimate=("cost_estimate", "mean"),
                mean_role_completeness=("role_completeness_proxy", "mean"),
            )
            .sort_values("accepted_rate", ascending=False)
        )

        ie_summary_df["cost_per_accepted_event"] = (
            ie_summary_df["mean_cost_estimate"] / ie_summary_df["accepted_rate"].clip(lower=1e-9)
        )

        display(ie_summary_df.round(4))
        print("IE detail sample:")
        display(ie_detail_df.head(12))

,variant,total_records,accepted_rate,mean_confidence,mean_tokens,mean_cost_estimate,mean_role_completeness,cost_per_accepted_event
0,baseline,34,1.0,0.9336,22.4412,22.4412,0.7471,22.4412
1,tokenization_bio_normalization,34,1.0,0.9331,22.3529,22.3529,0.7471,22.3529
2,tokenization_only,34,1.0,0.9331,22.3529,22.3529,0.7471,22.3529
3,tokenization_plus_bio,34,1.0,0.9336,22.4412,22.4412,0.7471,22.4412


IE detail sample:


,variant,original_sentence,processed_sentence,event_type,accepted,abstain_reason,confidence,tokens_processed,cost_estimate,role_completeness_proxy
0,baseline,Rockwell International Corp. 's Tulsa unit said it signed a tentative agreement extending its contract with Boeing Co. to provide structural parts for Boeing 's 747 jetliners .,Rockwell International Corp. 's Tulsa unit said it signed a tentative agreement extending its contract with Boeing Co. to provide structural parts for Boeing 's 747 jetliners .,other,True,None,0.954393,31,31.0,0.8
1,baseline,Rockwell said the agreement calls for it to supply 200 additional so-called shipsets for the planes .,Rockwell said the agreement calls for it to supply 200 additional so-called shipsets for the planes .,meeting,True,None,0.991064,18,18.0,0.8
2,baseline,"These include , among other parts , each jetliner 's two major bulkheads , a pressure floor , torque box , fixed leading edges for the wings and an aft keel beam .","These include , among other parts , each jetliner 's two major bulkheads , a pressure floor , torque box , fixed leading edges for the wings and an aft keel beam .",other,True,None,0.962927,28,28.0,0.6
3,baseline,"Under the existing contract , Rockwell said , it has already delivered 793 of the shipsets to Boeing .","Under the existing contract , Rockwell said , it has already delivered 793 of the shipsets to Boeing .",other,True,None,0.968601,16,16.0,0.8
4,baseline,"Rockwell , based in El Segundo , Calif. , is an aerospace , electronics , automotive and graphics concern .","Rockwell , based in El Segundo , Calif. , is an aerospace , electronics , automotive and graphics concern .",other,True,None,0.967011,16,16.0,0.8
5,baseline,"Frank Carlucci III was named to this telecommunications company 's board , filling the vacancy created by the death of William Sobey last May .","Frank Carlucci III was named to this telecommunications company 's board , filling the vacancy created by the death of William Sobey last May .",deadline,True,None,0.823199,24,24.0,0.8
6,baseline,"Mr. Carlucci , 59 years old , served as defense secretary in the Reagan administration .","Mr. Carlucci , 59 years old , served as defense secretary in the Reagan administration .",other,True,None,0.963921,14,14.0,0.8
7,baseline,"In January , he accepted the position of vice chairman of Carlyle Group , a merchant banking concern .","In January , he accepted the position of vice chairman of Carlyle Group , a merchant banking concern .",other,True,None,0.968942,16,16.0,0.8
8,baseline,SHEARSON LEHMAN HUTTON Inc .,SHEARSON LEHMAN HUTTON Inc .,other,True,None,0.618030,4,4.0,0.2
9,baseline,"Thomas E. Meador , 42 years old , was named president and chief operating officer of Balcor Co. , a Skokie , Ill. , subsidiary of this New York investment banking firm .","Thomas E. Meador , 42 years old , was named president and chief operating officer of Balcor Co. , a Skokie , Ill. , subsidiary of this New York investment banking firm .",other,True,None,0.964281,30,30.0,0.6


## 9. Error Analysis and Decision Gate

Summarize failure buckets and produce a recommendation on whether inference-time preprocessing is enough or retraining should be considered.

In [23]:
error_bucket_df = pd.DataFrame()
decision_df = pd.DataFrame()

if ablation_detail_df.empty:
    print("No ablation detail available for error analysis.")
else:
    mismatch_bucket = (
        ablation_detail_df.assign(bucket=np.where(ablation_detail_df["length_match"], "length_match", "length_mismatch"))
        .groupby(["variant", "bucket"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    low_alignment_bucket = (
        ablation_detail_df.assign(
            bucket=np.where(
                ablation_detail_df["alignment_coverage"] >= ALIGNMENT_MIN_COVERAGE,
                "alignment_ok",
                "alignment_low",
            )
        )
        .groupby(["variant", "bucket"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    low_conf_bucket = pd.DataFrame()
    if not ie_detail_df.empty:
        low_conf_bucket = (
            ie_detail_df.assign(
                bucket=np.where(
                    ie_detail_df["confidence"] >= 0.55,
                    "confidence_ok",
                    "confidence_low",
                )
            )
            .groupby(["variant", "bucket"], as_index=False)
            .size()
            .rename(columns={"size": "count"})
        )

    error_bucket_df = pd.concat([mismatch_bucket, low_alignment_bucket], ignore_index=True)
    if not low_conf_bucket.empty:
        error_bucket_df = pd.concat([error_bucket_df, low_conf_bucket], ignore_index=True)

    display(error_bucket_df)

    baseline_row = ablation_metrics_df[ablation_metrics_df["variant"] == "baseline"]
    best_row = ablation_metrics_df.sort_values("f1", ascending=False, na_position="last").head(1)

    baseline_f1 = float(baseline_row["f1"].iloc[0]) if not baseline_row.empty else np.nan
    best_f1 = float(best_row["f1"].iloc[0]) if not best_row.empty else np.nan
    best_variant = str(best_row["variant"].iloc[0]) if not best_row.empty else "none"
    best_alignment_evaluable = (
        float(best_row["alignment_evaluable_rate"].iloc[0])
        if not best_row.empty and "alignment_evaluable_rate" in best_row.columns
        else np.nan
    )

    f1_gain = best_f1 - baseline_f1 if (best_f1 == best_f1 and baseline_f1 == baseline_f1) else np.nan

    near_zero_gain = bool(f1_gain == f1_gain and f1_gain < 0.005)
    retraining_needed = bool((f1_gain != f1_gain) or near_zero_gain)

    if f1_gain != f1_gain:
        reason = "Could not compute reliable F1 gain after alignment-aware scoring; retraining is recommended."
    elif near_zero_gain:
        reason = "F1 gain remains near zero after alignment-aware scoring; retraining with tokenizer-aware labels is the next step."
    elif best_alignment_evaluable == best_alignment_evaluable and best_alignment_evaluable < 0.5:
        reason = "Alignment-aware evaluable coverage is still low; improve alignment first, then reassess retraining."
    else:
        reason = "Inference-time preprocessing shows measurable improvement; retraining is optional for now."

    decision_df = pd.DataFrame(
        [
            {
                "best_variant": best_variant,
                "baseline_f1": baseline_f1,
                "best_f1": best_f1,
                "f1_gain": f1_gain,
                "best_alignment_evaluable_rate": best_alignment_evaluable,
                "retraining_recommended": retraining_needed,
                "reason": reason,
            }
        ]
    )

    display(decision_df.round(4))

,variant,bucket,count
0,baseline,length_match,29
1,baseline,length_mismatch,221
2,tokenization_bio_normalization,length_match,29
3,tokenization_bio_normalization,length_mismatch,221
4,tokenization_only,length_match,29
5,tokenization_only,length_mismatch,221
6,tokenization_plus_bio,length_match,29
7,tokenization_plus_bio,length_mismatch,221
8,baseline,alignment_low,4
9,baseline,alignment_ok,246


,best_variant,baseline_f1,best_f1,f1_gain,best_alignment_evaluable_rate,retraining_recommended,reason
0,baseline,0.9562,0.9562,0.0,0.984,True,F1 gain remains near zero after alignment-aware scoring; retraining with tokenizer-aware labels is the next step.


## 10. Reproducibility Outputs and Final Recommendation

Persist compact artifacts so results can be reused in docs/dashboard updates without rerunning every analysis block.

In [34]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "domain-tokenization-study"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

saved_files = []

if not ablation_metrics_df.empty:
    path = OUTPUT_DIR / "ablation_metrics.csv"
    ablation_metrics_df.to_csv(path, index=False)
    saved_files.append(path)

if not ablation_detail_df.empty:
    path = OUTPUT_DIR / "ablation_detail.csv"
    ablation_detail_df.to_csv(path, index=False)
    saved_files.append(path)

if not ie_summary_df.empty:
    path = OUTPUT_DIR / "ie_summary.csv"
    ie_summary_df.to_csv(path, index=False)
    saved_files.append(path)

if not ie_detail_df.empty:
    path = OUTPUT_DIR / "ie_detail.csv"
    ie_detail_df.to_csv(path, index=False)
    saved_files.append(path)

if not error_bucket_df.empty:
    path = OUTPUT_DIR / "error_buckets.csv"
    error_bucket_df.to_csv(path, index=False)
    saved_files.append(path)

if not decision_df.empty:
    path = OUTPUT_DIR / "decision_gate.csv"
    decision_df.to_csv(path, index=False)
    saved_files.append(path)

_retrain_metrics_df = globals().get("retrain_metrics_df")
if _retrain_metrics_df is not None and not _retrain_metrics_df.empty:
    path = OUTPUT_DIR / "retrain_metrics.csv"
    _retrain_metrics_df.to_csv(path, index=False)
    saved_files.append(path)

_retrain_compare_df = globals().get("retrain_compare_df")
if _retrain_compare_df is not None and not _retrain_compare_df.empty:
    path = OUTPUT_DIR / "retrain_comparison.csv"
    _retrain_compare_df.to_csv(path, index=False)
    saved_files.append(path)

_retrain_sweep_df = globals().get("retrain_sweep_df")
if _retrain_sweep_df is not None and not _retrain_sweep_df.empty:
    path = OUTPUT_DIR / "retrain_sweep.csv"
    _retrain_sweep_df.to_csv(path, index=False)
    saved_files.append(path)

print("Saved files:")
for path in saved_files:
    print("-", path)

if _retrain_compare_df is not None and not _retrain_compare_df.empty:
    recommendation = _retrain_compare_df.iloc[0]["reason"]
    print("\nFinal recommendation:", recommendation)
elif not decision_df.empty:
    recommendation = decision_df.iloc[0]["reason"]
    print("\nFinal recommendation:", recommendation)

Saved files:
- /home/sg/dev/nlp/outputs/domain-tokenization-study/ablation_metrics.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/ablation_detail.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/ie_summary.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/ie_detail.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/error_buckets.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/decision_gate.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/retrain_metrics.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/retrain_comparison.csv
- /home/sg/dev/nlp/outputs/domain-tokenization-study/retrain_sweep.csv

Final recommendation: Best retraining result improves baseline, but gain is below +0.005 deployment threshold.


## 11. Tokenizer-Aware Retraining

Retrain a token-classification model using tokenizer-aware label alignment as the next step after inference-time preprocessing plateau.

This section:

- projects labels onto normalized tokens with alignment coverage checks
- aligns labels with tokenizer word_ids for training
- fine-tunes and evaluates on the same task metrics

In [25]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
from transformers.utils.notebook import NotebookProgressCallback


@dataclass(frozen=True)
class RetrainConfig:
    model_checkpoint: str = CFG.model_checkpoint
    output_dir: Path = PROJECT_ROOT / "outputs" / "domain-tokenization-retrain-distilbert"
    use_normalized_tokens: bool = True
    min_projection_coverage: float = 0.60
    max_train_samples: int = 3000
    max_test_samples: int = 1000
    num_train_epochs: float = 1.0
    train_batch_size: int = 16
    eval_batch_size: int = 16
    learning_rate: float = 2e-5


RETRAIN_CFG = RetrainConfig()
label2id = {label: idx for idx, label in enumerate(CHUNK_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}


def project_labels_to_tokens(
    source_tokens: Sequence[str],
    source_labels: Sequence[str],
    target_tokens: Sequence[str],
) -> Tuple[List[str], float]:
    """Project source labels onto target tokens using text alignment."""
    source_norm = [_normalize_alignment_token(tok) for tok in source_tokens]
    target_norm = [_normalize_alignment_token(tok) for tok in target_tokens]

    matcher = SequenceMatcher(a=source_norm, b=target_norm, autojunk=False)
    projected = ["O"] * len(target_tokens)
    covered_target = set()

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            continue
        block_len = min(i2 - i1, j2 - j1)
        for off in range(block_len):
            src_idx = i1 + off
            tgt_idx = j1 + off
            if src_idx >= len(source_labels):
                continue
            projected[tgt_idx] = source_labels[src_idx]
            covered_target.add(tgt_idx)

    coverage = len(covered_target) / max(len(target_tokens), 1)
    return fix_bio_sequence(projected), coverage


def build_retrain_frame(df: pd.DataFrame, cfg: RetrainConfig) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        raw_tokens = list(row["tokens"])
        raw_labels = list(row["chunk_labels"])

        if cfg.use_normalized_tokens:
            normalized_sentence = normalize_domain_text(sentence_from_tokens(raw_tokens))
            retrain_tokens = custom_word_tokenize(normalized_sentence)
            retrain_labels, projection_coverage = project_labels_to_tokens(
                raw_tokens,
                raw_labels,
                retrain_tokens,
            )
        else:
            retrain_tokens = raw_tokens
            retrain_labels = raw_labels
            projection_coverage = 1.0

        if projection_coverage < cfg.min_projection_coverage:
            continue

        rows.append(
            {
                "id": str(row["id"]),
                "tokens": retrain_tokens,
                "chunk_labels": retrain_labels,
                "chunk_tag_ids": [label2id.get(lbl, label2id["O"]) for lbl in retrain_labels],
                "projection_coverage": float(projection_coverage),
            }
        )

    return pd.DataFrame(rows)


retrain_train_df = build_retrain_frame(train_df, RETRAIN_CFG)
retrain_test_df = build_retrain_frame(test_df, RETRAIN_CFG)

if RETRAIN_CFG.max_train_samples is not None:
    retrain_train_df = retrain_train_df.head(min(RETRAIN_CFG.max_train_samples, len(retrain_train_df))).copy()
if RETRAIN_CFG.max_test_samples is not None:
    retrain_test_df = retrain_test_df.head(min(RETRAIN_CFG.max_test_samples, len(retrain_test_df))).copy()

print("Retrain rows (train/test):", len(retrain_train_df), len(retrain_test_df))
print("Mean projection coverage (train):", float(retrain_train_df["projection_coverage"].mean()))
print("Mean projection coverage (test):", float(retrain_test_df["projection_coverage"].mean()))

display(retrain_train_df[["tokens", "chunk_labels", "projection_coverage"]].head(3))

Retrain rows (train/test): 3000 1000
Mean projection coverage (train): 0.9948330462170513
Mean projection coverage (test): 0.9923917772113019


,tokens,chunk_labels,projection_coverage
0,"[Confidence, in, the, pound, is, widely, expected, to, take, another, sharp, dive, if, trade, figures, for, September, ,, due, for, release, tomorrow, ,, fail, to, show, a, substantial, improvemen...","[B-NP, B-PP, B-NP, I-NP, B-VP, I-VP, I-VP, I-VP, I-VP, B-NP, I-NP, I-NP, B-SBAR, B-NP, I-NP, B-PP, B-NP, O, B-ADJP, B-PP, B-NP, B-NP, O, B-VP, I-VP, I-VP, B-NP, I-NP, I-NP, B-PP, B-NP, I-NP, I-NP,...",1.0
1,"[Chancellor, of, the, Exchequer, Nigel, Lawson, 's, restated, commitment, to, a, firm, monetary, policy, has, helped, to, prevent, a, freefall, in, sterling, over, the, past, week, .]","[O, B-PP, B-NP, I-NP, B-NP, I-NP, B-NP, I-NP, I-NP, B-PP, B-NP, I-NP, I-NP, I-NP, B-VP, I-VP, I-VP, I-VP, B-NP, I-NP, B-PP, B-NP, B-PP, B-NP, I-NP, I-NP, O]",1.0
2,"[But, analysts, reckon, underlying, support, for, sterling, has, been, eroded, by, the, chancellor, 's, failure, to, announce, any, new, policy, measures, in, his, Mansion, House, speech, last, Th...","[O, B-NP, B-VP, B-NP, I-NP, B-PP, B-NP, B-VP, I-VP, I-VP, B-PP, B-NP, I-NP, B-NP, I-NP, B-VP, I-VP, B-NP, I-NP, I-NP, I-NP, B-PP, B-NP, I-NP, I-NP, I-NP, B-NP, I-NP, O]",1.0


In [ ]:
def to_hf_token_frame(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "id": df["id"].astype(str),
            "tokens": df["tokens"],
            "chunk_tag_ids": df["chunk_tag_ids"],
        }
    )


retrain_hf = DatasetDict(
    {
        "train": Dataset.from_pandas(to_hf_token_frame(retrain_train_df), preserve_index=False),
        "test": Dataset.from_pandas(to_hf_token_frame(retrain_test_df), preserve_index=False),
    }
)

tokenizer_kwargs = {}
if "roberta" in RETRAIN_CFG.model_checkpoint.lower():
    tokenizer_kwargs["add_prefix_space"] = True

retrain_tokenizer = AutoTokenizer.from_pretrained(RETRAIN_CFG.model_checkpoint, **tokenizer_kwargs)


def tokenize_and_align_labels_retrain(examples):
    tokenized = retrain_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    aligned_labels = []

    for i, label_seq in enumerate(examples["chunk_tag_ids"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        row_labels = []

        for word_idx in word_ids:
            if word_idx is None:
                row_labels.append(-100)
            elif word_idx != prev and word_idx < len(label_seq):
                row_labels.append(int(label_seq[word_idx]))
            else:
                row_labels.append(-100)
            prev = word_idx

        aligned_labels.append(row_labels)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_retrain_ds = retrain_hf.map(tokenize_and_align_labels_retrain, batched=True)


def compute_retrain_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_seq.append(CHUNK_LABELS[int(p)])
                l_seq.append(CHUNK_LABELS[int(l)])
        if p_seq and l_seq:
            true_preds.append(p_seq)
            true_labels.append(l_seq)

    if _SEQEVAL is None:
        total = sum(len(seq) for seq in true_labels)
        correct = sum(
            1
            for pred_seq, label_seq in zip(true_preds, true_labels)
            for pred_tag, label_tag in zip(pred_seq, label_seq)
            if pred_tag == label_tag
        )
        acc = correct / max(total, 1)
        return {
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "accuracy": acc,
        }

    out = _SEQEVAL.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": float(out.get("overall_precision", np.nan)),
        "recall": float(out.get("overall_recall", np.nan)),
        "f1": float(out.get("overall_f1", np.nan)),
        "accuracy": float(out.get("overall_accuracy", np.nan)),
    }


retrain_model = AutoModelForTokenClassification.from_pretrained(
    RETRAIN_CFG.model_checkpoint,
    num_labels=len(CHUNK_LABELS),
    id2label=id2label,
    label2id=label2id,
)

# Compatibility shim for accelerate versions that do not support keep_torch_compile.
import inspect
import accelerate

if "keep_torch_compile" not in inspect.signature(accelerate.Accelerator.unwrap_model).parameters:
    _orig_unwrap_model = accelerate.Accelerator.unwrap_model

    def _unwrap_model_compat(self, model, keep_torch_compile=None):
        _ = keep_torch_compile
        return _orig_unwrap_model(self, model)

    accelerate.Accelerator.unwrap_model = _unwrap_model_compat

retrain_args = TrainingArguments(
    output_dir=str(RETRAIN_CFG.output_dir),
    learning_rate=RETRAIN_CFG.learning_rate,
    per_device_train_batch_size=RETRAIN_CFG.train_batch_size,
    per_device_eval_batch_size=RETRAIN_CFG.eval_batch_size,
    num_train_epochs=RETRAIN_CFG.num_train_epochs,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

retrain_trainer = Trainer(
    model=retrain_model,
    args=retrain_args,
    train_dataset=tokenized_retrain_ds["train"],
    eval_dataset=tokenized_retrain_ds["test"],
    processing_class=retrain_tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=retrain_tokenizer),
    compute_metrics=compute_retrain_metrics,
)

try:
    retrain_trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

retrain_start = time.perf_counter()
retrain_trainer.train()
retrain_train_seconds = time.perf_counter() - retrain_start

retrain_metrics = retrain_trainer.evaluate()
retrain_metrics.update(
    {
        "train_seconds": float(retrain_train_seconds),
        "train_rows": int(len(retrain_train_df)),
        "test_rows": int(len(retrain_test_df)),
        "mean_projection_coverage_train": float(retrain_train_df["projection_coverage"].mean()),
        "mean_projection_coverage_test": float(retrain_test_df["projection_coverage"].mean()),
    }
)

retrain_metrics_df = pd.DataFrame([retrain_metrics])
display(retrain_metrics_df.round(4))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6614.27it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/sg/dev/nlp/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

,eval_loss,eval_precision,eval_recall,eval_f1,eval_accuracy,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,train_seconds,train_rows,test_rows,mean_projection_coverage_train,mean_projection_coverage_test
0,0.278,0.8776,0.8871,0.8823,0.9302,1.9137,522.56,32.921,1.0,22.1713,3000,1000,0.9948,0.9924


## 12. Retraining Comparison and Decision

Compare retrained model quality to the current baseline from the ablation table and emit a deployment recommendation.

In [33]:
retrain_compare_df = pd.DataFrame()

baseline_row = ablation_metrics_df[ablation_metrics_df["variant"] == "baseline"]
baseline_f1 = float(baseline_row["f1"].iloc[0]) if not baseline_row.empty else np.nan

candidates = []

if "retrain_metrics" in globals() and retrain_metrics:
    candidates.append(
        {
            "source": "single_retrain",
            "experiment": "single_retrain",
            "f1": float(retrain_metrics.get("eval_f1", np.nan)),
            "precision": float(retrain_metrics.get("eval_precision", np.nan)),
            "recall": float(retrain_metrics.get("eval_recall", np.nan)),
            "accuracy": float(retrain_metrics.get("eval_accuracy", np.nan)),
        }
    )

if "retrain_sweep_df" in globals() and isinstance(retrain_sweep_df, pd.DataFrame) and not retrain_sweep_df.empty:
    sweep_completed = retrain_sweep_df[retrain_sweep_df["status"] == "completed"].copy()
    if not sweep_completed.empty:
        sweep_completed = sweep_completed.sort_values(
            ["f1_gain_vs_baseline", "eval_f1"],
            ascending=[False, False],
            na_position="last",
        )
        best_sweep = sweep_completed.iloc[0]
        candidates.append(
            {
                "source": "iterative_sweep",
                "experiment": str(best_sweep.get("experiment", "unknown_sweep_trial")),
                "f1": float(best_sweep.get("eval_f1", np.nan)),
                "precision": float(best_sweep.get("eval_precision", np.nan)),
                "recall": float(best_sweep.get("eval_recall", np.nan)),
                "accuracy": float(best_sweep.get("eval_accuracy", np.nan)),
            }
        )

if not candidates:
    print("Retraining metrics are unavailable. Run retraining and/or sweep cells first.")
else:
    selected = max(candidates, key=lambda item: (item["f1"] == item["f1"], item["f1"]))

    retrain_f1 = float(selected["f1"])
    retrain_precision = float(selected["precision"])
    retrain_recall = float(selected["recall"])
    retrain_acc = float(selected["accuracy"])

    retrain_f1_gain = retrain_f1 - baseline_f1 if (retrain_f1 == retrain_f1 and baseline_f1 == baseline_f1) else np.nan

    retrain_recommended_for_deploy = bool(retrain_f1_gain == retrain_f1_gain and retrain_f1_gain >= 0.005)

    if retrain_f1_gain != retrain_f1_gain:
        retrain_reason = "Retraining metrics could not be computed reliably."
    elif retrain_f1_gain <= -0.005:
        retrain_reason = "Best retraining result underperforms baseline; do not deploy retrained variant."
    elif retrain_recommended_for_deploy:
        retrain_reason = "Best retraining result improves baseline by at least +0.005 F1 and should be considered for deployment."
    elif retrain_f1_gain > 0:
        retrain_reason = "Best retraining result improves baseline, but gain is below +0.005 deployment threshold."
    else:
        retrain_reason = "Retraining gain is near zero; revisit data, labels, or larger-model strategy."

    retrain_compare_df = pd.DataFrame(
        [
            {
                "baseline_f1": baseline_f1,
                "selected_retrain_source": selected["source"],
                "selected_experiment": selected["experiment"],
                "retrain_f1": retrain_f1,
                "retrain_precision": retrain_precision,
                "retrain_recall": retrain_recall,
                "retrain_accuracy": retrain_acc,
                "retrain_f1_gain": retrain_f1_gain,
                "deploy_retrained_model": retrain_recommended_for_deploy,
                "reason": retrain_reason,
            }
        ]
    )

    display(retrain_compare_df.round(4))

,baseline_f1,selected_retrain_source,selected_experiment,retrain_f1,retrain_precision,retrain_recall,retrain_accuracy,retrain_f1_gain,deploy_retrained_model,reason
0,0.9562,iterative_sweep,norm_off_e3_lr2e5_full,0.9582,0.9559,0.9605,0.9741,0.002,False,"Best retraining result improves baseline, but gain is below +0.005 deployment threshold."


## 13. Iterative Retraining Sweep

Run multiple stronger retraining configurations and keep iterating until either:

- a model beats baseline by at least +0.005 F1, or
- all configured trials are exhausted.

In [32]:
from dataclasses import replace
import gc

baseline_row = ablation_metrics_df[ablation_metrics_df["variant"] == "baseline"]
baseline_f1_for_sweep = float(baseline_row["f1"].iloc[0]) if not baseline_row.empty else np.nan

SWEEP_CONFIGS = [
    {
        "name": "norm_off_e2_lr2e5_t6000",
        "use_normalized_tokens": False,
        "max_train_samples": 6000,
        "max_test_samples": 1500,
        "num_train_epochs": 2.0,
        "learning_rate": 2e-5,
    },
    {
        "name": "norm_off_e3_lr2e5_full",
        "use_normalized_tokens": False,
        "max_train_samples": None,
        "max_test_samples": None,
        "num_train_epochs": 3.0,
        "learning_rate": 2e-5,
    },
    {
        "name": "norm_off_e3_lr1e5_full",
        "use_normalized_tokens": False,
        "max_train_samples": None,
        "max_test_samples": None,
        "num_train_epochs": 3.0,
        "learning_rate": 1e-5,
    },
]

retrain_sweep_rows = []

for idx, cfg_item in enumerate(SWEEP_CONFIGS, start=1):
    run_cfg = replace(
        RETRAIN_CFG,
        output_dir=PROJECT_ROOT / "outputs" / "domain-tokenization-retrain-sweep" / cfg_item["name"],
        use_normalized_tokens=cfg_item["use_normalized_tokens"],
        max_train_samples=cfg_item["max_train_samples"],
        max_test_samples=cfg_item["max_test_samples"],
        num_train_epochs=cfg_item["num_train_epochs"],
        learning_rate=cfg_item["learning_rate"],
    )

    run_train_df = build_retrain_frame(train_df, run_cfg)
    run_test_df = build_retrain_frame(test_df, run_cfg)

    if run_cfg.max_train_samples is not None:
        run_train_df = run_train_df.head(min(run_cfg.max_train_samples, len(run_train_df))).copy()
    if run_cfg.max_test_samples is not None:
        run_test_df = run_test_df.head(min(run_cfg.max_test_samples, len(run_test_df))).copy()

    if run_train_df.empty or run_test_df.empty:
        retrain_sweep_rows.append(
            {
                "experiment": cfg_item["name"],
                "status": "skipped_empty_data",
                "train_rows": int(len(run_train_df)),
                "test_rows": int(len(run_test_df)),
                "eval_f1": np.nan,
                "f1_gain_vs_baseline": np.nan,
            }
        )
        continue

    run_hf = DatasetDict(
        {
            "train": Dataset.from_pandas(to_hf_token_frame(run_train_df), preserve_index=False),
            "test": Dataset.from_pandas(to_hf_token_frame(run_test_df), preserve_index=False),
        }
    )

    local_tokenizer_kwargs = {}
    if "roberta" in run_cfg.model_checkpoint.lower():
        local_tokenizer_kwargs["add_prefix_space"] = True

    run_tokenizer = AutoTokenizer.from_pretrained(run_cfg.model_checkpoint, **local_tokenizer_kwargs)

    def _tokenize_and_align(examples):
        tok = run_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, label_seq in enumerate(examples["chunk_tag_ids"]):
            word_ids = tok.word_ids(batch_index=i)
            prev = None
            row = []
            for word_idx in word_ids:
                if word_idx is None:
                    row.append(-100)
                elif word_idx != prev and word_idx < len(label_seq):
                    row.append(int(label_seq[word_idx]))
                else:
                    row.append(-100)
                prev = word_idx
            all_labels.append(row)
        tok["labels"] = all_labels
        return tok

    run_tokenized_ds = run_hf.map(_tokenize_and_align, batched=True)

    run_model = AutoModelForTokenClassification.from_pretrained(
        run_cfg.model_checkpoint,
        num_labels=len(CHUNK_LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    run_args = TrainingArguments(
        output_dir=str(run_cfg.output_dir),
        learning_rate=run_cfg.learning_rate,
        per_device_train_batch_size=run_cfg.train_batch_size,
        per_device_eval_batch_size=run_cfg.eval_batch_size,
        num_train_epochs=run_cfg.num_train_epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
    )

    run_trainer = Trainer(
        model=run_model,
        args=run_args,
        train_dataset=run_tokenized_ds["train"],
        eval_dataset=run_tokenized_ds["test"],
        processing_class=run_tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=run_tokenizer),
        compute_metrics=compute_retrain_metrics,
    )

    try:
        run_trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass

    run_start = time.perf_counter()
    run_trainer.train()
    run_train_seconds = time.perf_counter() - run_start

    run_eval = run_trainer.evaluate()

    run_eval_f1 = float(run_eval.get("eval_f1", np.nan))
    run_gain = run_eval_f1 - baseline_f1_for_sweep if (run_eval_f1 == run_eval_f1 and baseline_f1_for_sweep == baseline_f1_for_sweep) else np.nan

    retrain_sweep_rows.append(
        {
            "experiment": cfg_item["name"],
            "status": "completed",
            "train_rows": int(len(run_train_df)),
            "test_rows": int(len(run_test_df)),
            "epochs": float(run_cfg.num_train_epochs),
            "learning_rate": float(run_cfg.learning_rate),
            "use_normalized_tokens": bool(run_cfg.use_normalized_tokens),
            "eval_precision": float(run_eval.get("eval_precision", np.nan)),
            "eval_recall": float(run_eval.get("eval_recall", np.nan)),
            "eval_f1": run_eval_f1,
            "eval_accuracy": float(run_eval.get("eval_accuracy", np.nan)),
            "train_seconds": float(run_train_seconds),
            "mean_projection_coverage_train": float(run_train_df["projection_coverage"].mean()),
            "mean_projection_coverage_test": float(run_test_df["projection_coverage"].mean()),
            "f1_gain_vs_baseline": run_gain,
        }
    )

    print(
        f"Sweep {idx}/{len(SWEEP_CONFIGS)} | {cfg_item['name']} | "
        f"eval_f1={run_eval_f1:.4f} | gain={run_gain:.4f}"
    )

    if run_gain == run_gain and run_gain >= 0.005:
        print("Target reached: F1 gain >= 0.005. Stopping sweep early.")
        del run_trainer, run_model
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        break

    del run_trainer, run_model
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

retrain_sweep_df = pd.DataFrame(retrain_sweep_rows)
if not retrain_sweep_df.empty and "f1_gain_vs_baseline" in retrain_sweep_df.columns:
    retrain_sweep_df = retrain_sweep_df.sort_values("f1_gain_vs_baseline", ascending=False, na_position="last")

display(retrain_sweep_df.round(4))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6897.96it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/sg/dev/nlp/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

Sweep 1/3 | norm_off_e2_lr2e5_t6000 | eval_f1=0.9469 | gain=-0.0093


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7650.21it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/sg/dev/nlp/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

Sweep 2/3 | norm_off_e3_lr2e5_full | eval_f1=0.9582 | gain=0.0020


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7101.77it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/sg/dev/nlp/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

Sweep 3/3 | norm_off_e3_lr1e5_full | eval_f1=0.9525 | gain=-0.0036


,experiment,status,train_rows,test_rows,epochs,learning_rate,use_normalized_tokens,eval_precision,eval_recall,eval_f1,eval_accuracy,train_seconds,mean_projection_coverage_train,mean_projection_coverage_test,f1_gain_vs_baseline
1,norm_off_e3_lr2e5_full,completed,8937,2013,3.0,0.0,False,0.9559,0.9605,0.9582,0.9741,179.1044,1.0,1.0,0.0020
2,norm_off_e3_lr1e5_full,completed,8937,2013,3.0,0.0,False,0.9484,0.9567,0.9525,0.9707,179.2776,1.0,1.0,-0.0036
0,norm_off_e2_lr2e5_t6000,completed,6000,1500,2.0,0.0,False,0.9406,0.9532,0.9469,0.9682,81.8448,1.0,1.0,-0.0093
